In [2]:
!pip install langchain langchain-core langchain-community langchain-google-genai langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 k

In [15]:
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata
from langchain_core.prompts import (
    PromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    ChatPromptTemplate
)
from langchain_core.output_parsers import StrOutputParser

# 1. Initialize the Chat Model
chat_model = ChatGoogleGenerativeAI(model="gemini-2.5-flash",
                                  temperature=0,
                                  google_api_key=userdata.get('GOOGLE_API_KEY'))
# 2. Prompt
instruction_str = """
### `###ROLE###`
You are a **Senior Patient Experience Analyst** specializing in healthcare quality and patient satisfaction. Your expertise lies in extracting specific, actionable insights from qualitative patient reviews to help stakeholders understand the patient journey.

### `###CONTEXT###`
The following context contains raw, unstructured patient reviews and feedback regarding their visits to a specific hospital facility. The goal is to provide users with a clear, fact-based understanding of the hospital's performance based strictly on these first-hand accounts.
**Context to analyze:** {context}

### `###TASK###`
Analyze the provided context to answer the user's inquiry by following these steps:
1. **Evidence Extraction:** Scan the reviews for direct mentions of the topics in the user's question.
2. **Synthesis:** Consolidate multiple perspectives (e.g., if three patients mention long wait times, summarize this as a recurring theme).
3. **Sentiment Calibration:** Maintain the tone of the original reviews—if patients were frustrated, reflect the seriousness of their concerns without being hyperbolic.
4. **Attribution:** Where possible, refer to the specific types of visits or departments mentioned (e.g., "Several ER patients noted...") to provide depth.

### `###CONSTRAINTS###`
* **Strict Grounding:** Do not use outside knowledge or general hospital statistics. If the information is not in the `{context}`, you must state: "I'm sorry, the provided reviews do not contain information regarding [Topic]."
* **No Fabrications:** Never invent patient names, specific dates, or medical outcomes not explicitly stated.
* **Neutrality:** Do not defend the hospital. Your job is to report what the patients said, whether positive or negative.
* **Professionalism:** Avoid using overly emotional language yourself; remain an objective analyst.

### `###EXAMPLES###`
**Input Context:** "The nursing staff in the maternity ward was incredible, but the billing department took three months to process my insurance."
**User Question:** "How is the administrative support at this hospital?"
**Response:** "Based on patient feedback, the administrative experience is mixed. While clinical staff (nurses) are highly praised for their care, there are reported delays in the billing department, with at least one patient citing a three-month delay in insurance processing."

### `###OUTPUT FORMAT###`
* Use **Markdown** for clarity.
* Use **bold text** for key themes or departments.
* If multiple points are addressed, use a **bulleted list**.
* End with a "Summary of Sentiment" sentence (e.g., "Overall, patients describe the experience as highly professional despite administrative hurdles").
"""

review_system_prompt = SystemMessagePromptTemplate(
    prompt=PromptTemplate(
        input_variables=["context"], template=instruction_str
    )
)

review_human_prompt = HumanMessagePromptTemplate(
    prompt=PromptTemplate(
        input_variables=["question"], template="Here's the user's question: {question}"
    )
)
messages = [review_system_prompt, review_human_prompt]

# This is our final, reusable prompt template
review_prompt_template = ChatPromptTemplate(
    input_variables=["context", "question"],
    messages=messages,
)


I'm sorry, the provided reviews do not contain information regarding positive experiences. The only feedback available states, "I had a negative stay!"


In [17]:
import time
from langchain_community.document_loaders.csv_loader import CSVLoader

REVIEWS_CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/Datasets/reviews.csv"
REVIEWS_CHROMA_PATH = "/content/drive/MyDrive/Colab Notebooks/chroma_data"

# Create an instance of the CSVLoader.
loader = CSVLoader(
    file_path=REVIEWS_CSV_PATH,  # Specify the path to the CSV file to be loaded.
    source_column="review"       # Specify the name of the column that contains the main text content.
)
reviews = loader.load()

In [19]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

embedding_function = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",  # Choose the specific embedding model provided by Google.
    google_api_key=userdata.get('GOOGLE_API_KEY')  # Securely fetch the Google API key.
)

batch_size = 20
num_batches = (len(reviews) - 1) // batch_size + 1
reviews_vector_db = None

# Loop through the documents in batches to avoid hitting the API's rate limit.
for i in range(0, len(reviews), batch_size):
    # Get the current batch of documents.
    batch_reviews = reviews[i:i + batch_size]
    current_batch_num = i // batch_size + 1

    print(f"Processing batch {current_batch_num}/{num_batches}...")

    if i == 0:
        # For the first batch, create a new Chroma vector database.
        # The `from_documents` method handles the entire process of embedding and storing the data.
        reviews_vector_db = Chroma.from_documents(
            documents=batch_reviews,  # Pass the list of Document objects that need to be embedded.
            embedding=embedding_function,
            # Specify the directory on the disk where the vector database will be saved.
            # This makes the database persistent, so we can load it directly in the future.
            persist_directory=REVIEWS_CHROMA_PATH
        )
    else:
        # For subsequent batches, add the documents to the existing database.
        reviews_vector_db.add_documents(documents=batch_reviews)

    # Pause the script for 30 seconds after each batch to respect the per-minute rate limit.
    print(f"Batch {current_batch_num} processed. Waiting for 60 seconds to avoid per-minute rate limits...")
    time.sleep(60)

print("Vector database created successfully and saved to the specified directory.")

Processing batch 1/51...
Batch 1 processed. Waiting for 60 seconds to avoid per-minute rate limits...
Processing batch 2/51...
Batch 2 processed. Waiting for 60 seconds to avoid per-minute rate limits...
Processing batch 3/51...
Batch 3 processed. Waiting for 60 seconds to avoid per-minute rate limits...
Processing batch 4/51...
Batch 4 processed. Waiting for 60 seconds to avoid per-minute rate limits...
Processing batch 5/51...
Batch 5 processed. Waiting for 60 seconds to avoid per-minute rate limits...
Processing batch 6/51...
Batch 6 processed. Waiting for 60 seconds to avoid per-minute rate limits...
Processing batch 7/51...
Batch 7 processed. Waiting for 60 seconds to avoid per-minute rate limits...
Processing batch 8/51...
Batch 8 processed. Waiting for 60 seconds to avoid per-minute rate limits...
Processing batch 9/51...
Batch 9 processed. Waiting for 60 seconds to avoid per-minute rate limits...
Processing batch 10/51...
Batch 10 processed. Waiting for 60 seconds to avoid per-

In [20]:
reviews_vector_db = Chroma(persist_directory=REVIEWS_CHROMA_PATH, embedding_function=embedding_function)

In [21]:
from langchain_core.runnables import RunnablePassthrough

# Create a retriever to fetch the top 10 most relevant reviews based on a query
reviews_retriever = reviews_vector_db.as_retriever(k=10)
# The `as_retriever` method converts the database into a retriever.
# `k=10` specifies that the retriever should return the top 10 most relevant documents for a query.

# Create a chain for querying and generating responses
review_chain = (
    {"context": reviews_retriever, "question": RunnablePassthrough()}
    # Step 1: Retrieves relevant reviews (`context`) and passes the `question` unchanged
    | review_prompt_template
    # Step 2: Formats the retrieved reviews and the user's question into a structured prompt
    | chat_model
    # Step 3: Sends the prompt to the OpenAI chat model to generate a response
    | StrOutputParser()
    # Step 4: Parses the model's raw output into a clean string format for easier use
)

In [22]:
!pip install gradio

In [ ]:
import gradio as gr

# Function that will run when user asks a question
def ask_question(question):
    response = review_chain.invoke(question)
    return response

# Create Gradio interface
interface = gr.Interface(
    fn=ask_question,                 # Function to call
    inputs=gr.Textbox(
        lines=2,
        placeholder="Ask something about hospital reviews..."
    ),
    outputs="text",
    title="Healthcare RAG Intelligence System",
    description="Ask questions about hospital reviews using RAG."
)

# Launch UI
interface.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8218dd9223781c7e95.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
